In [38]:
from pathlib import Path
import csv
import spacy
from spacy.matcher import PhraseMatcher
import pandas as pd
import numpy as np
import tarfile
from collections import defaultdict
from collections import Counter
import json
import os

In [ ]:
'''
spacy.cli.download("en_core_web_sm")
'''

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 38.7 MB/s eta 0:00:00 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
filename = "data/metadata.csv"
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

parser = spacy.load("en_core_web_sm", disable=["ner"])

# Terms and their categories
terms = [
    "registry", "registries", "database", "databases", "platform", "platforms",
    "repository", "repositories", "ipd-ma", "data dashboard",
    "individual participant data meta-analysis", "multicenter cohort",
    "multi-centre cohort", "patient-level", "patient level" "participant level",
    "participant-level", "individual participant data", "meta-analysis"
]

term_categories = {
    "registry": "infrastructure",
    "registries": "infrastructure",
    "database": "infrastructure",
    "databases": "infrastructure",
    "platform": "infrastructure",
    "platforms": "infrastructure",
    "repository": "infrastructure",
    "repositories": "infrastructure",
    "data dashboard": "infrastructure",
    "ipd-ma": "study_type",
    "individual participant data meta-analysis": "study_type",
    "multicenter cohort": "study_type",
    "multi-centre cohort": "study_type",
    "patient-level": "data_level",
    "patient level": "data_level",
    "participant level": "data_level",
    "participant-level": "data_level",
    "individual participant data": "data_level",
    "meta-analysis": "study_type"
}

phrase_matcher = PhraseMatcher(parser.vocab, attr="LOWER")
phrase_matcher.add("TERMS", [parser.make_doc(term) for term in terms])


def get_appositive_phrases(doc):
    """Extract full appositive phrases around matched terms."""
    phrases = []
    for _, start, end in phrase_matcher(doc):
        span = doc[start:end]
        if any(tok.dep_ == "appos" for tok in span):
            # Find the appos token and its head
            for tok in span:
                if tok.dep_ == "appos":
                    head = tok.head
                    # Small window around head + appos
                    start_idx = max(head.i - 3, 0)
                    end_idx = min(tok.i + 1, len(doc))
                    phrase = doc[start_idx:end_idx].text.strip()
                    phrases.append(phrase)
    return sorted(set(phrases), key=lambda s: s.lower())


def get_trigger_terms(doc):
    triggers = []
    for _, start, end in phrase_matcher(doc):
        span = doc[start:end]
        if any(tok.dep_ == "appos" for tok in span):
            triggers.append(span.text)
    return sorted(set(triggers), key=str.lower)


rows = []
seen_titles = set()

with open(filename, encoding="utf-8") as f_in:
    reader = csv.DictReader(f_in)

    titles = []
    uids = []
    dois = []
    sha_list = []
    pdf_json_list = []
    pmc_json_list = []

    for row in reader:
        title = (row.get("title") or "").strip()
        cord_uid = (row.get("cord_uid") or "").strip()
        doi = (row.get("doi") or "").strip()
        sha = (row.get("sha") or "").strip()
        pdf_json_files = (row.get("pdf_json_files") or "").strip()
        pmc_json_files = (row.get("pmc_json_files") or "").strip()

        if not title or title in seen_titles:
            continue
        seen_titles.add(title)

        titles.append(title)
        uids.append(cord_uid)
        dois.append(doi)
        sha_list.append(sha)
        pdf_json_list.append(pdf_json_files)
        pmc_json_list.append(pmc_json_files)


docs = parser.pipe(titles, batch_size=1000)

for idx, (cord_uid, doc) in zip(range(len(uids)), zip(uids, docs)):
    trigger_terms = get_trigger_terms(doc)
    if not trigger_terms:
        continue

    appositive_phrases = get_appositive_phrases(doc)

    rows.append({
        "cord_uid": cord_uid,
        "doi": dois[idx],
        "sha": sha_list[idx],
        "pdf_json_files": pdf_json_list[idx],
        "pmc_json_files": pmc_json_list[idx],
        "title": doc.text,
        "trigger_terms": "; ".join(trigger_terms),
        "appositive_phrases": "; ".join(appositive_phrases)
    })

# Save only hits
out_path = output_dir / "appos_with_metadata.csv"
with open(out_path, "w", encoding="utf-8", newline="") as f_out:
    writer = csv.DictWriter(
        f_out,
        fieldnames=[
            "cord_uid",
            "doi",
            "sha",
            "pdf_json_files",
            "pmc_json_files",
            "title",
            "trigger_terms",
            "appositive_phrases"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

In [34]:
df = pd.read_csv("output/appos_with_metadata.csv")

print(len(df))
display(df)

1626


,cord_uid,doi,sha,pdf_json_files,pmc_json_files,title,trigger_terms,appositive_phrases
0,02opdk0m,10.1093/nar/gkp278,b411e12b20d883ef2ee5ca19d48eff9fccedf05f,document_parses/pdf_json/b411e12b20d883ef2ee5c...,document_parses/pmc_json/PMC2703908.xml.json,CVTree update: a newly designed phylogenetic s...,platform,CVTree update: a newly designed phylogenetic s...
1,b66bb2ri,10.1093/nar/gkq1013,f7c0f747be302193f876a3d3ddf1e14098090e75,document_parses/pdf_json/f7c0f747be302193f876a...,document_parses/pmc_json/PMC3013642.xml.json,PhEVER: a database for the global exploration ...,database,PhEVER: a database
2,ppzxvykb,10.1093/nar/gkr1064,2a17719f2c1d211a651060441bda8bc1c052e2aa,document_parses/pdf_json/2a17719f2c1d211a65106...,document_parses/pmc_json/PMC3245074.xml.json,ELM—the database of eukaryotic linear motifs,database,ELM—the database
3,s50g754b,10.3390/s120201648,7b00b9f04112c671d8c31227d31b3fea7ca10c7e,document_parses/pdf_json/7b00b9f04112c671d8c31...,document_parses/pmc_json/PMC3304132.xml.json,Protein Reporter Bioassay Systems for the Phen...,Platform,Protein Reporter Bioassay Systems for the Phen...
4,nebf6u0z,10.1371/journal.pone.0042972,bf0e3ed6e8fa4adbfacbf76e9fa2f5aa1ccdef19,document_parses/pdf_json/bf0e3ed6e8fa4adbfacbf...,document_parses/pmc_json/PMC3434151.xml.json,The VNTR Polymorphism of the DC-SIGNR Gene and...,Meta-Analysis,The VNTR Polymorphism of the DC-SIGNR Gene and...
...,...,...,...,...,...,...,...,...
1621,b7jasqjh,10.3389/fphys.2020.00937,f7c2a3ee167a67ed7e9c9d00aaed90eb0c80da33,document_parses/pdf_json/f7c2a3ee167a67ed7e9c9...,document_parses/pmc_json/PMC7487972.xml.json,Metabolic Imaging and Biological Assessment: P...,Platforms,Metabolic Imaging and Biological Assessment: P...
1622,7j6ci5xk,10.18240/ijo.2021.10.19,NaN,NaN,NaN,Visual differences in topography-guided versus...,Meta-analysis,Visual differences in topography-guided versus...
1623,xm01nj2i,10.1007/s10620-022-07401-2,bc3240a5bd1d5764b7c35f8fbfb43edd2d3da5c3,document_parses/pdf_json/bc3240a5bd1d5764b7c35...,document_parses/pmc_json/PMC8853241.xml.json,Prevalence and Influencing Factors of Irritabl...,Meta-Analysis,Prevalence and Influencing Factors of Irritabl...
1624,tg06vxza,10.1186/1471-2105-14-95,3eb3c6212432888ecf0f2d3d2dc4a2fb6b4f17c0,document_parses/pdf_json/3eb3c6212432888ecf0f2...,document_parses/pmc_json/PMC3636126.xml.json,CGAP: a new comprehensive platform for the com...,platform,CGAP: a new comprehensive platform


In [3]:
n_with_fulltext = (
    (df["pdf_json_files"].notna() & (df["pdf_json_files"] != "")) |
    (df["pmc_json_files"].notna() & (df["pmc_json_files"] != ""))
).sum()

print("Total hits:", df.shape[0])

print("Rows with pmc_json_files:", df["pmc_json_files"].notna().sum())
print("Rows with pdf_json_files:", df["pdf_json_files"].notna().sum())

print("Rows with at least one full text:", n_with_fulltext)
print("Rows with no full text:", df.shape[0] - n_with_fulltext)

Total hits: 1626
Rows with pmc_json_files: 488
Rows with pdf_json_files: 617
Rows with at least one full text: 628
Rows with no full text: 998


In [34]:
df[["pdf_json_files", "pmc_json_files"]].head()

,pdf_json_files,pmc_json_files
0,document_parses/pdf_json/b411e12b20d883ef2ee5c...,document_parses/pmc_json/PMC2703908.xml.json
1,document_parses/pdf_json/f7c0f747be302193f876a...,document_parses/pmc_json/PMC3013642.xml.json
2,document_parses/pdf_json/2a17719f2c1d211a65106...,document_parses/pmc_json/PMC3245074.xml.json
3,document_parses/pdf_json/7b00b9f04112c671d8c31...,document_parses/pmc_json/PMC3304132.xml.json
4,document_parses/pdf_json/bf0e3ed6e8fa4adbfacbf...,document_parses/pmc_json/PMC3434151.xml.json


UNPACK FULL TEXTS

In [ ]:
tar_path = "data/document_parses.tar.gz"
output_dir = "data/document_parses_extracted_subset"

os.makedirs(output_dir, exist_ok=True)

# collect all filenames from df
needed_files = set()

for row in df.to_dict("records"):
    for col in ["pmc_json_files", "pdf_json_files"]:
        cell = row[col]
        if cell is None or cell == "" or cell == "nan":
            continue
        # Split by ';' if multiple paths
        parts = str(cell).split(";")
        for p in parts:
            p = p.strip()
            if p:
                needed_files.add(p)

print(f"Total unique JSON files needed: {len(needed_files)}")

# extract those files from the tar
with tarfile.open(tar_path, "r:gz") as tar:
    for member in tar.getmembers():
        if member.name in needed_files:
            target_path = os.path.join(output_dir, member.name)
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            with tar.extractfile(member) as src, open(target_path, "wb") as dst:
                dst.write(src.read())
            needed_files.remove(member.name)  # optional: track progress

print("Extraction complete.")

Total unique JSON files needed: 1153
Extraction complete.


What's actually inside these pdf & pmc files? what's the structure? what sections? etc.

In [18]:
def read_json_from_file(json_path):
    full_path = os.path.join("data/document_parses_extracted_subset", json_path)
    try:
        with open(full_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

In [11]:
row = df[df["pdf_json_files"].notna()].iloc[0]

path = row["pdf_json_files"]
print(path)

json_data = read_json_from_file(path)

print(json_data is None)

json_data.keys()

document_parses/pdf_json/b411e12b20d883ef2ee5ca19d48eff9fccedf05f.json
False


dict_keys(['paper_id', 'metadata', 'abstract', 'body_text', 'bib_entries', 'ref_entries', 'back_matter'])

In [12]:
row = df[df["pmc_json_files"].notna()].iloc[0]

path = row["pmc_json_files"]
print(path)

json_data = read_json_from_file(path)

print(json_data is None)

json_data.keys()

document_parses/pmc_json/PMC2703908.xml.json
False


dict_keys(['paper_id', 'metadata', 'body_text', 'ref_entries', 'back_matter', 'bib_entries'])

Check first how many have non-empty back matter

In [37]:
non_empty_backs = []

count_non_empty = 0
count_total_checked = 0


def iter_paths(cell):
    if cell is None or cell == "" or cell == "nan":
        return []
    parts = str(cell).split(";")
    return [p.strip() for p in parts if p.strip()]


for row_idx, row in enumerate(df.to_dict("records")):
    json_data = None

    pmc_paths = iter_paths(row["pmc_json_files"])
    pdf_paths = iter_paths(row["pdf_json_files"])

    all_paths = pmc_paths + pdf_paths

    for json_filename in all_paths:
        data = read_json_from_file(json_filename)
        if data is not None:
            json_data = data
            break

    if json_data is None:
        continue

    count_total_checked += 1

    back = json_data.get("back_matter", [])

    if isinstance(back, list) and len(back) > 0:
        count_non_empty += 1
        non_empty_backs.append((row_idx, json_filename, back))


print("Checked:", count_total_checked)
print("Non-empty back_matter:", count_non_empty)

Checked: 628
Non-empty back_matter: 85


-> this is good, because there are exactly 628 rows with at least one text

In [40]:
sample_size = 5
for i, (row_idx, json_filename, back) in enumerate(non_empty_backs[:sample_size]):
    print(f"=== Example {i+1} (row {row_idx}, file {json_filename}) ===")
    print(json.dumps(back, indent=2)[:2000])  # first 2000 chars
    print()

=== Example 1 (row 17, file document_parses/pdf_json/30f9f8dd4536eea9368e9214f216cf40917eddb5.json) ===
[
  {
    "text": "No financial or nonfinancial benefits have been received or will be received from any party related directly or indirectly to the subject of this article.",
    "cite_spans": [],
    "ref_spans": [],
    "section": "Conflict of interest"
  }
]

=== Example 2 (row 54, file document_parses/pdf_json/8c5b16990e135cea6c6e6a10d5db42bd4d69cbfc.json) ===
[
  {
    "text": "The authors declare that they have no known competing financial interests or personal relationships that could have appeared to influence the work reported in this paper.",
    "cite_spans": [],
    "ref_spans": [],
    "section": "acknowledgement"
  }
]

=== Example 3 (row 76, file document_parses/pdf_json/88ef0f6e75ed7ac93b9c9b8e68811fd2f7b2f660.json) ===
[
  {
    "text": "This research was supported by the New Strategic Research (P2P) Project Fiscal Year 2022, Walailak University, Thailand.",
    "ci

In [41]:
section_types = []
for _, _, back in non_empty_backs:
    for item in back:
        if isinstance(item, dict):
            section_types.append(item.get("section", item.get("type", "unknown")))

print("Section type distribution in back_matter:")
for t, c in Counter(section_types).most_common(20):
    print(f"{t}: {c}")

Section type distribution in back_matter:
Acknowledgements: 20
(which was not certified by peer review): 20
annex: 19
acknowledgement: 10
Acknowledgments: 9
ACKNOWLEDGEMENTS: 6
Conflict of interest: 3
Author contributions: 3
FUNDING: 3
Author Contributions: 3
8,9: 3
12: 3
Competing interests: 2
Supplementary Figures: 2
ACKNOWLEDGMENTS: 2
ACKNOWLEDGEMENT: 2
Data and Code Availability: 2
Declaration of interests: 2
Acknowledgment: 2
Funding: 2


Extract data availabiltiy statements from body of text

In [42]:

def extract_das_from_json(json_data):
    """
    Extract data availability statement text from JSON dict.

    Searches both: body_text & back_matter

    Returns a string with the DAS text (or None if not found).
    """
    if json_data is None:
        return None

    das_keywords = [
        "data availability",
        "availability of data",
        "availability of data and materials",
        "materials and data",
        "data and materials",
        "data access",
        "availability of data, code, and other materials",
        "data and code availability",
        "code and data availability",
        "supporting data",
        "code availability"
    ]

    def extract_from_section(section_list):
        texts = []

        if not isinstance(section_list, list):
            return texts

        for paragraph_dict in section_list:
            section_name = paragraph_dict.get("section", "").lower()
            paragraph_text = paragraph_dict.get("text", "")

            if not paragraph_text:
                continue

            if any(keyword in section_name for keyword in das_keywords):
                texts.append(paragraph_text.strip())

        return texts

    das_texts = []


    # Search body
    das_texts.extend(extract_from_section(json_data.get("body_text", [])))

    # Search back matter
    das_texts.extend(extract_from_section(json_data.get("back_matter", [])))

    if not das_texts:
        return None

    return "\n".join(das_texts)

def get_das_text_for_row(row, tar):
    # Try pmc first
    pmc_path_raw = row.get("pmc_json_files", "")
    pdf_path_raw = row.get("pdf_json_files", "")

    # Helper to iterate over semicolon-separated paths
    def iterate_paths(path_str):
        if pd.isna(path_str) or not path_str:
            return []
        # Split by semicolon and strip whitespace
        parts = [p.strip() for p in path_str.split(";") if p.strip()]
        return parts

    # Try PMC paths
    for path in iterate_paths(pmc_path_raw):
        # Ensure it starts with "document_parses/"
        if not path.startswith("document_parses/"):
            path = "document_parses/" + path
        json_data = read_json_from_file(path)
        das_text = extract_das_from_json(json_data)
        if das_text:
            return das_text

    # Try PDF paths
    for path in iterate_paths(pdf_path_raw):
        if not path.startswith("document_parses/"):
            path = "document_parses/" + path
        json_data = read_json_from_file(path)
        das_text = extract_das_from_json(json_data)
        if das_text:
            return das_text

    return None


In [43]:
# run function
df["data_availability_statement"] = [
    get_das_text_for_row(row, tar) for row in df.to_dict("records")
]

df["data_availability_statement"].notna().sum()

169

In [44]:
das_df = df.loc[
    df["data_availability_statement"].notna(),
    ["cord_uid", "doi", "title", "data_availability_statement"]
]

display(das_df.head())

das_df.to_csv("output/with_DAS.csv", index=False)

,cord_uid,doi,title,data_availability_statement
11,bbupvms3,10.1186/s12931-020-1287-4,Neuromuscular blocking agents for acute respir...,All data generated or analyzed during the pres...
12,hceyy9w7,10.1186/s13054-020-2765-2,Validation of neuromuscular blocking agent use...,The datasets used and/or analyzed during the c...
19,p3mb7r0v,10.1186/s13643-020-01401-x,Predicting the treatment response of certolizu...,The data that support the findings of this stu...
22,zer1m4b6,10.1186/s12913-020-05903-1,Incident reports involving hospital administra...,We used JCQHC open data. The datasets used and...
23,41i9h3vq,10.1186/s12931-020-01574-y,The efficacy of mesenchymal stromal cell-deriv...,The datasets used and/or analysed in this stud...


Check all ways 'data availability' sections could be mentioned

-> search for 'data' across ALL sections - the idea is to maybe use the result of this to add to the list manually of sections to extract (in the previous code chunk)

In [ ]:
section_counter = Counter()

for row in df.to_dict("records"):

    for column in ["pmc_json_files", "pdf_json_files"]:

        path_str = row.get(column, "")

        if pd.isna(path_str) or not path_str:
            continue

        paths = [p.strip() for p in path_str.split(";") if p.strip()]

        for path in paths:

            json_data = read_json_from_file(path)

            if json_data is None:
                continue

            for paragraph in json_data.get("body_text", []):

                section = paragraph.get("section", "")

                if "data" in section.lower():
                    section_counter[section] += 1

print(f"Found {len(section_counter)} unique section names.\n")

for section, count in section_counter.most_common():
    print(f"{count:3}  {section}")

Found 338 unique section names.

 17  Data Analysis
 13  Data extraction
 13  Data source
 12  DATA AVAILABILITY
 12  Data Processing
 11  INPUT DATA
 11  Data Extraction
 11  Data analysis
 11  Survey and review of global platforms on health data
 11  Misinformation Focused Datasets
 10  DATA AVAILABILITY STATEMENT
  9  Data Acquisition
  9  Data queried via users' Web Browsers
  8  Data extraction and quality assessment
  8  Data Extraction and Quality Assessment
  8  Data extraction ::: Methods
  8  Data Availability Statement
  8  Data source ::: Methods and materials
  8  Big data in healthcare ::: Big data and healthcare data interoperability
  8  Semantic integration ::: Healthcare interoperability ::: Big data and healthcare data interoperability
  8  Big data in healthcare
  7  . The MiREDiBase Search module. Users can filter out MiREDiBase data by exploiting the specific modal box (A). Then, they can dig into the data by interacting with the filtered editing sites (B). The ed